# Carlisle 2005: the River Eden flood against a full shallow-water reference

This notebook runs and analyses the real-river test of FloodA5: the January 2005
flood of the River Eden at Carlisle, where the Caldew and the Petteril join it.
It is the third of three notebooks, after `01_planar` (synthetic slopes) and
`02_channel` (a straight channel).

**The question.** How close is FloodA5's default **local-inertial** solver to the
**full shallow-water (SWE) reference** solver on a real river and floodplain? And
does the answer depend on the mesh? The event runs on two meshes of about the
same cell size:

- **A5 resolution 18**: pentagons of about 493 m², the area of a 22 m square;
- **Cartesian 20 m**: square cells aligned with the LiDAR, averaged to 20 m.

The **sub-grid (SGS)** solver also runs on A5.

**Two cases from the case registry** (`cases/registry/carlisle.jl`;
`docs/TEST_CASES.md` §3):

| Case | Forcing | Compared at | Used for |
|---|---|---|---|
| `carlisle_peak` | the three river inflows only | the 6 h before the Eden's peak, 49.5–55.5 h | **the main comparison** (§5) |
| `carlisle` | the inflows **plus 10 mm/h of uniform rain**, an artificial test forcing | the first 6 h | the recorded test of HYDRAULICS §15.10–15.11 (§6) |

In both cases the river starts dry.

The 10 mm/h rain of `carlisle` is **not the 2005 event's rain**. It is a test
load from the solver-damping series (HYDRAULICS §15.6–15.11): 60 mm over the whole
domain in 6 h. Early in the event the rivers carry little water, so most of what
looks flooded at 6 h in that case is a rain-on-grid sheet over the floodplain,
not river flooding. §6 shows this directly.

**How to run it.** Use the kernel *Julia FloodA5 1.12* (`notebooks/README.md`)
and run all cells.

- **First run.** The notebook builds the meshes (about 7 min) and makes the runs
  in `test/case_runs/carlisle/` and `test/case_runs/carlisle_peak/`:
  - `carlisle`: about 50 min, with one 4-thread run at a time;
  - `carlisle_peak`: about 1.5 h for each of the two SWE runs over 55.5 h, at
    6 threads each, and about 15–25 min for each of the others.
- **Later runs.** A run is reused when its output records the same command line
  and a completed run, so the notebook then takes a few minutes.
- **Analysis only.** `RUN_CASES = false` analyses the existing runs without
  starting any.

## Setup

The notebook loads three modules, all from the repository:

| Module | File | Purpose |
|---|---|---|
| `FA` | `analysis/FloodA5Analysis.jl` | loads runs and computes metrics |
| `FC` | `cases/FloodA5Cases.jl` | the case registry and runner |
| `FP` | `visualisation/FloodA5Plots.jl` | static CairoMakie figures |

None of them loads the model or Python. The simulations run as separate
`julia FloodModel.jl …` processes. The repository root comes from the active
project (the kernel starts Julia with `--project=<repo>`), so the notebook works
from any checkout.

In [ ]:
const REPO = dirname(Base.active_project())
include(joinpath(REPO, "analysis", "FloodA5Analysis.jl"));   const FA = FloodA5Analysis
include(joinpath(REPO, "cases", "FloodA5Cases.jl"));         const FC = FloodA5Cases
include(joinpath(REPO, "visualisation", "FloodA5Plots.jl")); const FP = FloodA5Plots
using CairoMakie, Printf, Statistics, Logging
import ArchGDAL as AG
CairoMakie.activate!(type = "png")   # PNG: an SVG of 30 000 cell polygons would be huge
FP.activate!()                       # the FloodA5 figure style

RUN_CASES = true      # false: analyse existing runs only (an error if one is missing)
PARALLEL  = 1         # simulations at a time
THREADS   = nothing   # threads per simulation; nothing = each run's own (4; 6 for the long SWE runs)

c6 = FC.case("carlisle")          # first 6 h, inflows + 10 mm/h test rain
cpk = FC.case("carlisle_peak")     # 0–55.5 h, inflows only; compared over 49.5–55.5 h
CARL = joinpath(REPO, "test", "carlisle")        # the tracked inputs
T6 = 6 * 3600.0                                  # the end of `carlisle`
T_WIN = (49.5, 52.5, 55.5) .* 3600.0             # the `carlisle_peak` comparison times
T_PEAK = T_WIN[end]                              # the Eden's peak, 55.5 h
hrs(t) = @sprintf("%.1f h", t / 3600)

# Small HTML tables (an IJulia Markdown table also carries a LaTeX copy that VS Code cannot render)
function htmltable(header, rows; caption = "")
    h(x) = replace(string(x), "&" => "&amp;", "<" => "&lt;", ">" => "&gt;")
    io = IOBuffer()
    print(io, "<table>")
    isempty(caption) || print(io, "<caption style='text-align:left'>", h(caption), "</caption>")
    print(io, "<thead><tr>", join(("<th>" * h(x) * "</th>" for x in header)), "</tr></thead><tbody>")
    for r in rows
        print(io, "<tr>", join(("<td style='text-align:right'>" * h(x) * "</td>" for x in r)), "</tr>")
    end
    print(io, "</tbody></table>")
    return HTML(String(take!(io)))
end
f3(x) = isfinite(x) ? @sprintf("%.3f", x) : "–"
f2(x) = isfinite(x) ? @sprintf("%.2f", x) : "–"
println("repository: ", REPO)

## 1. The cases in the registry

In the registry, a case is plain data:

- the preparation steps (here, mesh generation);
- the runs, each one `FloodModel.jl` argument list;
- the metrics computed from the runs;
- the physical checks the metrics must pass;
- expected values for regression (`cases/expected/<case>.toml`).

`FC.show_case` prints the case's description and every command exactly as the
runner executes it, so the text and the commands cannot drift apart. Each command
is what you would type from the repository root, with paths expanded for this
checkout. `carlisle_peak` reuses the meshes that `carlisle` prepares.

In [ ]:
FC.show_case(cpk)

In [ ]:
FC.show_case(c6)

## 2. The inputs

The cases use four tracked files in `test/carlisle/`:

| File | Content |
|---|---|
| `Carlisle_domain.geojson` | the area of interest (AOI): a 4.75 km × 3.06 km rectangle in British National Grid, stored in WGS 84 |
| `Carlisle_LiDAR_5m_mean.tif` | 5 m LiDAR terrain, British National Grid with ODN heights |
| `carlisle_wgs84.bci` | the boundary conditions: where water enters (LISFLOOD-FP format) |
| `carlisle_wgs84.bdy` | the inflow hydrographs the `.bci` refers to |

FloodA5 works in longitude and latitude on the sphere. The DEM stays in its own
coordinate system: the model reprojects its sample points into it, not the raster.

### 2.1 Domain and DEM

In [ ]:
aoi = read(joinpath(CARL, "Carlisle_domain.geojson"), String)
println("AOI properties: ", match(r"(\x22MINX[^}]*?)\s*\}", aoi).captures[1])

AG.read(joinpath(CARL, "Carlisle_LiDAR_5m_mean.tif")) do ds
    gt = AG.getgeotransform(ds)
    band = AG.getband(ds, 1)
    z = Float64.(AG.read(band)); nd = AG.getnodatavalue(band)
    v = filter(x -> isfinite(x) && (nd === nothing || x != nd), z)
    @printf("DEM: %d × %d pixels of %.0f m; elevation %.1f to %.1f m (median %.1f m)\n",
            AG.width(ds), AG.height(ds), gt[2], minimum(v), maximum(v), median(v))
    println("CRS: ", match(r"^\w+\[\x22([^\x22]+)\x22", AG.getproj(ds)).captures[1])
end

### 2.2 River inflows

The `.bci` file has one line per boundary condition:

- **`P` lines** are point inflows: longitude, latitude, the code `QVAR` (a
  discharge that varies in time) and the name of the hydrograph in the `.bdy`
  file. The model adds each hydrograph to the cell nearest its point. The three
  points are the River Eden (`main`), the Caldew (`cummersdale`) and the
  Petteril (`harraby`).
- **The `W` line** is a LISFLOOD-FP "west edge" boundary. FloodA5 domains can be
  any polygon, so it ignores `N`/`E`/`S`/`W` lines with a warning (`--bc-file`
  places boundaries by location instead).

FloodA5 does not need the `W` line: every edge of the domain is already an open
boundary, a free outflow at normal depth (`--open-boundary normal`, the default
since 0.3.0; HYDRAULICS §15.11).

The `.bdy` file holds one block per hydrograph: the name, the number of rows,
the time unit, then rows of *value, time* (in that order: discharge first).
The cell below reads it with the model's own reader,
`boundaryinputs/timeseries_io.jl`.

In [ ]:
println(read(joinpath(CARL, "carlisle_wgs84.bci"), String))
println(join(first(readlines(joinpath(CARL, "carlisle_wgs84.bdy")), 7), "\n"), "\n…")

module BDY   # the model's .bdy reader, on its own
    include(joinpath(Main.REPO, "boundaryinputs", "timeseries_io.jl"))
end
series = with_logger(ConsoleLogger(stderr, Logging.Warn)) do
    BDY.read_timeseries(BDY.LisfloodBDYReader(joinpath(CARL, "carlisle_wgs84.bdy")))
end
RIVERS = ("main", "cummersdale", "harraby")
RIVER = Dict("main" => "Eden (main)", "cummersdale" => "Caldew (cummersdale)", "harraby" => "Petteril (harraby)")
qat(n, t) = (s = series[n]; k = findfirst(>=(t - 1), s[1]); s[2][k])
vol_to(n, T) = (t = series[n][1]; q = series[n][2]; sum((q[k] + q[k+1]) / 2 * (t[k+1] - t[k]) for k in 1:length(t)-1 if t[k+1] <= T + 1))
htmltable(["river", "Q(0)", "Q(6 h)", "Q(49.5 h)", "Q(55.5 h)", "peak", "peak at", "volume 0–6 h", "volume 0–55.5 h"],
          [[RIVER[n], f2(qat(n, 0)), f2(qat(n, T6)), f2(qat(n, T_WIN[1])), f2(qat(n, T_PEAK)),
            f2(maximum(series[n][2])), hrs(series[n][1][argmax(series[n][2])]),
            f3(vol_to(n, T6) / 1e6), f3(vol_to(n, T_PEAK) / 1e6)] for n in RIVERS];
          caption = "Inflow hydrographs (discharge in m³/s, volumes in Mm³)")

In [ ]:
let fig = Figure(size = (900, 420))
    ax = Axis(fig[1, 1]; xlabel = "Time since the start of the .bdy series (h)", ylabel = "Discharge (m³/s)",
              title = "River inflows, 0–60 h", subtitle = "shaded: the two comparison windows")
    vspan!(ax, [0.0, T_WIN[1] / 3600], [T6 / 3600, T_PEAK / 3600]; color = [(FP.SERIES[4], 0.25), (FP.SERIES[1], 0.18)])
    for (i, n) in enumerate(RIVERS)
        t, q = series[n]; s = t .<= 60 * 3600
        lines!(ax, t[s] ./ 3600, q[s]; color = FP.SERIES[i], linewidth = 2, label = RIVER[n])
    end
    text!(ax, 3.0, 1180; text = "carlisle\n0–6 h\n+ test rain", align = (:center, :top), fontsize = 11, color = FP.INK2)
    text!(ax, 52.5, 650; text = "carlisle_peak\nwindow\n49.5–55.5 h", align = (:center, :top), fontsize = 11, color = FP.INK2)
    xlims!(ax, 0, 60); ylims!(ax, 0, 1350)
    Legend(fig[2, 1], ax; orientation = :horizontal)
    fig
end

The two windows see very different rivers:

- **The first 6 h (`carlisle`).** The rising limb has barely started. The Eden
  goes from about 100 to 230 m³/s, while the Caldew and Petteril stay roughly
  steady.
- **The 6 h before the peak (`carlisle_peak`).** The Eden rises from about
  1 030 m³/s to its peak of 1 273 m³/s at 55.5 h. The Caldew and Petteril peaked
  earlier, at 38.5 and 44 h, and are receding.

The model has no start-time option, so `carlisle_peak` simulates the whole event
from t = 0, with the river initially dry, and is compared only in the window.

**Rain.** `carlisle_peak` has none. `carlisle` adds `--rainfall 10`, a uniform
10 mm/h on every cell for the whole 6 h. As explained above, this is a test load,
not the observed rain; §6 shows how much of that case's flooding it makes.

## 3. The meshes

`carlisle` prepares three meshes in `test/case_runs/carlisle/`; `carlisle_peak`
uses the same ones. A preparation step is skipped when its output files exist.

### 3.1 A5 resolution 18 (pya5)

`FloodModel.jl --meshgen AOI --meshres 18 --dem DEM --meshout MESH --mesh-only`
builds the mesh in three stages. It runs with `--threads 1` because of a
PyCall/GDAL hazard.

1. **Julia samples the AOI.** It lays a dense grid of points over the AOI's
   bounding box, at 0.45 of a cell width, and keeps the points inside the
   polygon. This point-in-polygon test runs on the GPU if one is available,
   otherwise on CPU threads.
2. **Python finds the cells** (`mesh/a5_bridge.py`, run with FloodA5's Python:
   PyCall's Conda environment, where pya5 is installed). In order:
   - pya5's `polygon_to_cells` (pya5 1.x) gives every resolution-18 cell that
     overlaps the AOI;
   - the set grows by each cell's neighbours whose polygon intersects the AOI,
     until nothing changes;
   - two cells are neighbours if they share two vertices, which is exact, unlike
     pya5's `grid_disk`;
   - each cell's area comes from its vertices.

   The bridge writes a GeoParquet file and records the cell-ID scheme
   (`a5-v1`) in its metadata.
3. **Julia samples the DEM.** Each cell's elevation is the mean of the DEM,
   interpolated bilinearly at up to 256 quasi-random (Halton) points inside the
   polygon (`A5Grid.sample_dem_mean!`). Cells with no DEM under them get NaN and
   take no part in the flow.

The core of stage 2:

In [ ]:
# Print a few lines of a source file, from the first line that matches `from`
function excerpt(file, from, n)
    L = readlines(joinpath(REPO, file))
    i = findfirst(l -> occursin(from, l), L)
    i === nothing && return println("(", file, ": ", repr(from), " not found)")
    println(file, ", lines ", i, "–", i + n - 1, ":\n")
    foreach(println, L[i:i+n-1])
end
excerpt("mesh/a5_bridge.py", "# ── Pass 1: seed cell set", 16)
println()
excerpt("mesh/a5_bridge.py", "# ── Pass 2: grid_disk expansion", 26)

### 3.2 SGS tables on the same A5 mesh

The SGS solver needs more than one elevation per cell. The SGS mesh is the same
A5 mesh, generated with `--flow-model sgs`, so that `A5Grid.build_sgs_tables!`
also stores two kinds of sub-grid table from the DEM.

- **Per cell:** 512 Halton points give a 100-bin hypsometric curve, the stored
  volume and wetted area as functions of water level.
- **Per shared edge:** the DEM is sampled every 2.5 m along the edge, half the
  DEM pixel size. This gives the edge's sill elevation and its flow-area and
  wetted-perimeter curves.

The SGS solver measures depth from the cell's lowest DEM point and fills the
cell's low parts first, such as a channel narrower than the cell. Flow between
two cells is limited by the edge's own profile.

### 3.3 Cartesian 20 m

`cases/scripts/average_dem.jl` averages the 5 m LiDAR onto a 20 m grid
(`gdalwarp -tr 20 20 -r average -tap`: each 20 m cell is the mean of its 16
pixels, on a grid snapped to multiples of 20 m). `--meshgen-cartesian` then
turns each pixel into a cell whose elevation is the pixel value. This step needs
no Python.

The A5 cells are about 23% larger in area than the Cartesian cells, so the two
meshes have similar, but not equal, resolution.

The next cell runs the missing preparation steps. It prints "up to date" for
steps whose outputs exist. Then it prints the key lines of each step's log.

In [ ]:
FC.prepare(c6)
println()
keylines = r"Sample points:|Points inside AOI|Python bridge:|Mesh generated in|outside DEM extent|mean sampling complete|SGS pre-processing:|edge sample spacing|SGS tables built in \d|Raster-aligned Cartesian grid:|cells of \d+ m"
for p in c6.prep
    log = joinpath(FC.case_dir(c6), "prep_$(p.name).log")
    isfile(log) || continue
    println("── ", p.name)
    for l in eachline(log)
        occursin(keylines, l) || continue
        l = replace(l, r"[A-Za-z]:[\\/][^\s:]*[\\/]" => "", r"^[┌└│\[ ]*(Info:|Warning:)\s*" => "")
        println("   ", strip(l))
    end
end

The figure shows the two meshes' elevations, read from their GeoParquet files.
The zoom covers the Caldew joining the Eden:

- the A5 cells follow no axis, and a channel crosses them at every angle;
- the Cartesian grid staircases every channel that is not aligned with x or y.

The narrow Caldew and Petteril are only one or two cells wide on both meshes.

In [ ]:
a5_mesh   = FA.load_mesh_parquet(joinpath(FC.case_dir(c6), "carlisle_a5_res18.parquet"))
cart_mesh = FA.load_mesh_parquet(joinpath(FC.case_dir(c6), "carlisle_cart20.parquet"))
for (name, m) in (("A5 res 18", a5_mesh), ("Cartesian 20 m", cart_mesh))
    @printf("%-15s %6d cells, mean area %5.1f m² (a %.1f m square), domain %.2f km², %d cells without elevation\n",
            name, length(m), mean(m.area), sqrt(mean(m.area)), sum(m.area) / 1e6, count(isnan, m.elevation))
end

# A map of a mesh alone: FP draws runs, so wrap the mesh in a one-frame Run
mesh_run(m, label) = FA.Run("", label, 2, [0.0], m, Dict{String,Any}(), "")
let fig = Figure(), rng = (8.0, 32.0), box = (-2.9480, -2.9380, 54.8985, 54.9045)
    for (j, r) in enumerate((mesh_run(a5_mesh, "A5 res 18"), mesh_run(cart_mesh, "Cartesian 20 m")))
        FP.fieldmap!(fig[1, j], r, 1; var = :elevation, colormap = :batlowW, colorrange = rng, sources = false,
                     basemap = :none, title = r.label, subtitle = "cell elevation, $(length(r.mesh)) cells",
                     width = 430, colorbar = j == 2)
        p = FP.fieldmap!(fig[2, j], r, 1; var = :elevation, colormap = :batlowW, colorrange = rng, sources = false,
                         basemap = :none, title = r.label * ": zoom", subtitle = "the Caldew joining the Eden",
                         width = 430, colorbar = false)
        limits!(p.axis, box...)
        p.axis.aspect = AxisAspect((box[2] - box[1]) * cosd(54.9) / (box[4] - box[3]))
    end
    resize_to_layout!(fig)
    fig
end

## 4. The runs and the checks

Both cases make the same five runs; `carlisle` adds two runs without rain.

| Run | Mesh | Solver | Role |
|---|---|---|---|
| `a5_swe` | A5 | `--flow-model swe` | the reference on A5 (see below) |
| `a5_li` | A5 | `--flow-model standard` | the 0.3.0 local-inertial default; compared with `a5_swe` |
| `cart_swe` | Cartesian | `--flow-model swe` | the reference on Cartesian |
| `cart_li` | Cartesian | `--flow-model standard` | local-inertial; compared with `cart_swe` |
| `a5_sgs` | A5 + SGS tables | `--flow-model sgs` | the sub-grid solver |
| `a5_li_norain`, `cart_li_norain` | A5, Cartesian | `--flow-model standard` | `carlisle` only: the same 6 h without the test rain |

**The SWE reference** solves the full 2-D shallow-water equations, advection
included. It is a second-order HLLC finite-volume scheme with hydrostatic
reconstruction (HYDRAULICS §15.10). It runs on the same mesh, bed, sources and
boundaries as the local-inertial solver, so the comparison isolates what the
local-inertial simplification does on that mesh. In the 6 h test it converges in
its own numerical parameters: halving its Courant number or adding eddy viscosity
changes its storage by less than 0.1%.

**The SGS run** has no reference with a sub-grid bed, so it is compared with the
others by storage and water level only.

**Options:**

| Option | `carlisle_peak` | `carlisle` |
|---|---|---|
| `--inflow-bci` | the three hydrographs | the same |
| `--rainfall` | none | `10` (10 mm/h on every cell; test forcing) |
| `--sim-duration` | `199800` (55.5 h) | `21600` (6 h) |
| `--dt-max` | `10` (adaptive step, at most 10 s) | `10` |
| `--output-interval` | `1800` (a frame every 30 min) | `900` (every 15 min) |

The runner appends `--output test/case_runs/<case>/runs/<run>.h5` and writes a
log next to it. Every output file records its command
(`FO.run_info(path)["argv"]`), and `julia --project=. FloodModel.jl --rerun <run>.h5`
reproduces a run bit for bit.

The next cells print the commands, then run both cases: they make the runs that
are missing or out of date, analyse them and check them. With `RUN_CASES = false`
they only analyse and check the existing runs.

Each case's checks are listed at the end of its `FC.show_case` output above.
Every metric is also compared with `cases/expected/<case>.toml`, a regression
test. A5 results are sensitive to round-off in the bed. A mesh regenerated on
another machine can move them beyond those tight tolerances while the physical
checks still pass; if so, re-record the expected values
(`cases/run.jl baseline <case>`).

In [ ]:
for c in (cpk, c6), j in c.jobs
    println(c.name, " / ", j.name, "   (", join(("$k = $v" for (k, v) in j.tags), ", "), ")")
    println("  julia --project=. --threads ", j.threads, " FloodModel.jl ", join(FC.job_argv(c, j), " "), "\n")
end

In [ ]:
t0 = time()
ok6 = RUN_CASES ? FC.run_case(c6; parallel = PARALLEL, threads = THREADS) : FC.check(c6)   # also prepares the meshes
println()
okp = RUN_CASES ? FC.run_case(cpk; parallel = PARALLEL, threads = THREADS) : FC.check(cpk)
@printf("\n%s (%.1f min)\n", ok6 && okp ? "All checks pass." : "SOME CHECKS FAIL: see above.", (time() - t0) / 60)

In [ ]:
m6 = FC.analyse(c6; io = devnull)
mp = FC.analyse(cpk; io = devnull)
load_case(c, label) = Dict(j.name => FA.load_run(FC.job_output(c, j); label = label[j.name]) for j in c.jobs)
LABEL = Dict("a5_swe" => "A5 SWE", "a5_li" => "A5 local-inertial", "cart_swe" => "Cartesian SWE",
             "cart_li" => "Cartesian local-inertial", "a5_sgs" => "A5 SGS",
             "a5_li_norain" => "A5 local-inertial, no rain", "cart_li_norain" => "Cartesian local-inertial, no rain")
P = load_case(cpk, Dict(k => v * " (river only)" for (k, v) in LABEL))
S = load_case(c6, Dict(k => v * (endswith(k, "norain") ? "" : " (+ test rain)") for (k, v) in LABEL))
ORDER = ["a5_swe", "a5_li", "cart_swe", "cart_li", "a5_sgs"]

# run cost and volumes, from each output file's end-of-run record
cost(R, n) = (s = R[n].info["summary"]; [n, f3(s["vol_added_m3"] / 1e6), f3(s["vol_removed_m3"] / 1e6),
              @sprintf("%.1e", abs(s["conservation_error_m3"]) / s["vol_applied_m3"]), string(s["n_steps"]),
              @sprintf("%.1f", s["wall_simulation_s"] / 60)])
hdr = ["run", "added (Mm³)", "removed (Mm³)", "conservation error (rel.)", "steps", "wall (min)"]
display(htmltable(hdr, [cost(P, n) for n in ORDER]; caption = "carlisle_peak (0–55.5 h, river only): volumes at 55.5 h and cost"))
display(htmltable(hdr, [cost(S, n) for n in vcat(ORDER, ["a5_li_norain", "cart_li_norain"])];
                  caption = "carlisle (0–6 h; all but *_norain with the 10 mm/h test rain): volumes at 6 h and cost"))

How to read these tables:

- **Mass.** The conservation error, |added − stored − removed| ÷ added, is at
  round-off level in every run.
- **Cost.** The SWE runs take many times the steps of the local-inertial runs on
  the same mesh, because their stable step is bounded by the wave speed
  |u| + √(gh) rather than by √(gh) alone. The `wall` column is the simulation
  time of each run as it was made, on a shared machine.

## 5. The 6 h before the Eden's peak (`carlisle_peak`: river inflow only, 49.5–55.5 h)

### 5.1 Storage and flooded area

The plot shows the water stored in the domain at each 30-minute frame, over the
whole run; the comparison window is shaded. Colour marks the mesh (A5 blue,
Cartesian orange) and line style the solver (local-inertial solid, SWE dashed,
SGS dotted). The SGS storage is the sum of the cell volumes (`FA.storage`).

In [ ]:
PRUNS = [P[n] for n in ("a5_li", "a5_swe", "a5_sgs", "cart_li", "cart_swe")]
let fig = Figure(size = (820, 480))
    ax = FP.storage_plot!(fig[1, 1], PRUNS; title = "Water stored, carlisle_peak (river inflow only, no rain)")
    vspan!(ax, [T_WIN[1] / 3600], [T_PEAK / 3600]; color = (FP.MUTED, 0.18))
    text!(ax, (T_WIN[1] + T_PEAK) / 7200, 0; text = "window", align = (:center, :bottom), offset = (0, 6),
          fontsize = 11, color = FP.INK2)
    fig
end

In [ ]:
rows = [vcat([P[n].label], [f3(mp[n]["storage_$(t)_m3"] / 1e6) for t in ("w0", "wmid", "w1")],
             [f2(mp[n]["wet_area_0p1_$(t)_km2"]) for t in ("w0", "wmid", "w1")]) for n in ORDER]
display(htmltable(["run", "storage 49.5 h", "52.5 h", "55.5 h", "area > 0.1 m, 49.5 h", "52.5 h", "55.5 h"], rows;
                  caption = "carlisle_peak: storage (Mm³) and flooded area deeper than 0.1 m (km²) in the window"))
rows = [vcat([P[n].label], [f3(mp[n]["storage_ratio_$t"]) for t in ("w0", "wmid", "w1")],
             [f3(mp[n]["rms_dh_$(t)"]) for t in ("w0", "wmid", "w1")],
             [f2(mp[n]["csi_$t"]) for t in ("w0", "wmid", "w1")], [f2(mp[n]["bias_$t"]) for t in ("w0", "wmid", "w1")])
        for n in ("a5_li", "cart_li")]
display(htmltable(["run", "storage ÷ SWE 49.5 h", "52.5 h", "55.5 h", "rms Δh (m) 49.5 h", "52.5 h", "55.5 h",
                   "CSI 49.5 h", "52.5 h", "55.5 h", "bias 49.5 h", "52.5 h", "55.5 h"], rows;
                  caption = "carlisle_peak: local-inertial against SWE on the same mesh (FA.compare: rms over cells wetter than 1 cm in either; CSI and bias at 0.1 m)"))
# Mean speed where the reference is at least 3 m deep (FA.compare's speed_ratio): not a case metric
rows = Vector{String}[]
for (ref, run) in (("a5_swe", "a5_li"), ("cart_swe", "cart_li")), t in T_WIN
    c = FA.compare(P[ref], P[run]; k_ref = FA.frame_at(P[ref], t; tol = 300))
    push!(rows, [P[run].label, hrs(t), f2(c.speed_ref), f2(c.speed), f2(c.speed_ratio), string(c.n_deep)])
end
htmltable(["run", "time", "SWE speed (m/s)", "local-inertial speed (m/s)", "ratio", "cells ≥ 3 m deep in SWE"], rows;
          caption = "carlisle_peak: mean speed over the cells at least 3 m deep in the reference")

**Near the peak the result is the opposite of the 6 h test.** In the window both
local-inertial runs store *less* than the reference, not more.

| | A5 | Cartesian |
|---|---|---|
| local-inertial storage ÷ SWE | 0.74–0.76 | about 0.85 |
| storage at 55.5 h, local-inertial vs SWE (Mm³) | 8.01 vs 10.51 | 7.92 vs 9.36 |
| CSI | 0.86–0.87 | 0.92–0.93 |
| bias | 0.87–0.89 | 0.93–0.94 |

- **The local-inertial runs let more water out.** By 55.5 h the A5
  local-inertial run has let out 149.6 Mm³, against 147.1 Mm³ for the A5
  reference. That 2.5 Mm³ is the storage gap.
- **They flood less.** Their flooded area is 4.57 km² on both meshes, against
  5.16 (A5 SWE) and 4.91 km² (Cartesian SWE). A bias below 1 means the
  local-inertial run has fewer wet cells than the reference.
- **Their deep water moves faster** than the reference's: 1.3–1.4 times as fast
  on A5, and 1.15 times on Cartesian.
- **The mesh matters for the reference, not for local-inertial.** The two
  local-inertial runs are nearly identical (8.01 and 7.92 Mm³, 4.57 km² each),
  while the two references differ by 12% (10.51 and 9.36 Mm³). The A5 reference's
  deep water is also the slowest of the four runs (about 1.1 m/s, against 1.5 m/s
  for the Cartesian reference).
- **The gap is not only at the peak.** The storage plot shows the local-inertial
  runs below their references from early in the event on A5, and from about 10 h
  on Cartesian.

### 5.2 Depth maps in the window: local-inertial against SWE

Depth at 49.5, 52.5 and 55.5 h (river inflow only). Cells shallower than 0.1 m
show the terrain. The stars mark the three inflow points.

In [ ]:
DMAX = 7.0   # one colour range for all the peak-window maps (m)
FP.fieldmaps([P["a5_li"], P["a5_swe"]], collect(T_WIN); dry_threshold = 0.1, colorrange = (0, DMAX), panel = 330)

In [ ]:
FP.fieldmaps([P["cart_li"], P["cart_swe"]], collect(T_WIN); dry_threshold = 0.1, colorrange = (0, DMAX), panel = 330)

At this scale the four floods look alike. In every run the Eden fills its
floodplain between the meanders, and the Caldew and Petteril valleys flood. The
differences are in depth (§5.3) and at the edges of the flood. One example is the
floodplain inside the Eden's north-western loop: the A5 reference floods it from
about 32 h and the Cartesian reference from about 46 h, but the local-inertial
runs only inside the window (§5.4).

### 5.3 Differences at the peak (55.5 h)

The two upper maps show local-inertial minus SWE depth at 55.5 h, on cells wetter
than 1 cm in either run, so their rms is the rms Δh of the table. Blue means the
local-inertial run is deeper. The extent maps classify each cell at the 0.1 m
threshold as wet in both (a hit), wet in the reference only (a miss), or wet in
the local-inertial run only (a false alarm).

In [ ]:
let fig = Figure()
    FP.difference_map!(fig[1, 1], P["a5_swe"], P["a5_li"]; t = T_PEAK, colorrange = (-1.5, 1.5), width = 460,
                       title = "A5: local-inertial − SWE, 55.5 h, river only")
    FP.difference_map!(fig[1, 2], P["cart_swe"], P["cart_li"]; t = T_PEAK, colorrange = (-1.5, 1.5), width = 460,
                       title = "Cartesian: local-inertial − SWE, 55.5 h, river only")
    resize_to_layout!(fig)
    fig
end

In [ ]:
let fig = Figure()
    FP.extent_map!(fig[1, 1], P["a5_swe"], P["a5_li"]; t = T_PEAK, width = 430)
    FP.extent_map!(fig[2, 1], P["cart_swe"], P["cart_li"]; t = T_PEAK, width = 430)
    resize_to_layout!(fig)
    fig
end

Where is the extra (or missing) water? The next cell splits the storage
difference Σ (h − h_SWE)·A by the reference's depth in each cell.

In [ ]:
function storage_by_class(ref, run, t)
    kr, kk = FA.frame_at(ref, t), FA.frame_at(run, t)
    href, h, A = FA.depth(ref, kr), FA.depth(run, kk), ref.mesh.area
    rows = Vector{String}[]
    for (name, lo, hi) in (("dry or < 0.1 m in SWE", -Inf, 0.1), ("0.1–1 m", 0.1, 1.0),
                           ("1–3 m", 1.0, 3.0), ("> 3 m (river channels)", 3.0, Inf), ("all cells", -Inf, Inf))
        sel = [i for i in eachindex(h) if isfinite(h[i]) && isfinite(href[i]) && lo < href[i] <= hi]
        push!(rows, [name, string(length(sel)), @sprintf("%+.3f", sum((h[i] - href[i]) * A[i] for i in sel; init = 0.0) / 1e6),
                     isempty(sel) ? "–" : @sprintf("%+.3f", mean(h[sel] .- href[sel]))])
    end
    return htmltable(["SWE depth class", "cells", "Δ storage (Mm³)", "mean Δh (m)"], rows;
                     caption = run.label * " − " * ref.label * " at " * FP.hhmm(run.t[kk]))
end
display(storage_by_class(P["a5_swe"], P["a5_li"], T_PEAK))
display(storage_by_class(P["cart_swe"], P["cart_li"], T_PEAK))

**The deficit is in water that the reference already holds.** Almost none of it
is on cells that are dry in the reference. By SWE depth class:

| SWE depth class | A5 | Cartesian |
|---|---|---|
| floodplain 1–3 m deep (the largest share) | −1.64 Mm³, 0.54 m shallower on average | −0.96 Mm³, 0.34 m shallower |
| channels deeper than 3 m | −0.45 Mm³, 0.59 m shallower | −0.19 Mm³, 0.30 m shallower |
| shallow edges (0.1–1 m) | −0.41 Mm³ | −0.29 Mm³ |

**The local-inertial water surface is lower over the whole flood:** by about
half a metre on A5 and a third of a metre on Cartesian. The difference maps show
this as a near-uniform red over the floodplain, darkest in A5's north-western
loop. The local-inertial run is deeper only in small patches, such as the
Cartesian Eden at the western outflow edge and parts of the Caldew and Petteril
valleys.

### 5.4 Time series at a river cell and a floodplain point

Two points, marked on the 55.5 h map:

- **River Eden:** a point in the channel at the southern end of its central meanders.
- **Floodplain:** a point inside the Eden's north-western meander loop, which
  floods during the window.

The time series cover the whole `carlisle_peak` run (river inflow only), with the
comparison window shaded. Each run uses its own cell nearest each point. The SGS run is left out, because
its depth is measured from each cell's lowest point and so is not comparable with
the others.

In [ ]:
POINTS = [(lon = -2.93647, lat = 54.90020, label = "River Eden channel"),
          (lon = -2.95000, lat = 54.90400, label = "Floodplain in the north-western loop")]
foreach(p -> @printf("%-34s lon %.5f  lat %.5f\n", p.label, p.lon, p.lat), POINTS)
let fig = Figure()
    p = FP.fieldmap!(fig[1, 1], P["a5_swe"], nothing; t = T_PEAK, dry_threshold = 0.1, colorrange = (0, DMAX), width = 560)
    FP.mark_points!(p, POINTS; labels = [q.label for q in POINTS])
    resize_to_layout!(fig)
    fig
end

In [ ]:
let fig = FP.timeseries([P[n] for n in ("a5_li", "a5_swe", "cart_li", "cart_swe")], POINTS;
                        labels = ["A5 local-inertial", "A5 SWE", "Cartesian local-inertial", "Cartesian SWE"])
    for ax in filter(x -> x isa Axis, fig.content)
        vspan!(ax, [T_WIN[1] / 3600], [T_PEAK / 3600]; color = (FP.MUTED, 0.18))
    end
    display(fig)
end
rows = [vcat([P[n].label], [f2(FA.depth(P[n], FA.frame_at(P[n], t))[FA.source_cell(P[n].mesh, p.lon, p.lat)])
                            for p in POINTS for t in (T_WIN[1], T_PEAK)]) for n in ("a5_li", "a5_swe", "cart_li", "cart_swe")]
htmltable(["run", "Eden 49.5 h", "Eden 55.5 h", "floodplain 49.5 h", "floodplain 55.5 h"], rows;
          caption = "carlisle_peak: depth (m) at the two points")

**In the Eden channel** all four runs fill in the first 2–3 h and then follow the
hydrograph:

- The A5 reference stands highest throughout: about 1 m above the others for
  most of the event, and 0.3–0.6 m at the peak (7.67 m against 7.37 m for the
  Cartesian reference and about 7.1 m for both local-inertial runs).
- The two local-inertial runs stay within a few centimetres to a few decimetres
  of each other.

**On the floodplain inside the north-western loop** the bank is overtopped at
different times:

| Run | Bank overtopped at | Depth at 55.5 h |
|---|---|---|
| A5 reference | about 32 h | 1.84 m |
| Cartesian reference | about 46 h | 1.15 m |
| both local-inertial runs | about 50 h, inside the window | 0.71–0.85 m |

The A5 reference's higher river stage reaches the floodplain first.

### 5.5 The SGS run at the peak

At 55.5 h the SGS run stores 26.7 Mm³: 3.3 times the A5 local-inertial run and
2.5 times the A5 reference. The storage plot (§5.1) shows it pulling away from
the other runs from the first hours. The next cell checks whether its water is
held in the cells' low parts, below the cell-mean bed that the other solvers use,
or is higher water.

The map shows the SGS water surface minus the A5 local-inertial one at 55.5 h, on
the cells deeper than 0.1 m in both runs; the two runs are on the same cells. It
is limited to those cells because an SGS depth is measured from the cell's lowest
point: nearly every SGS cell holds a little water there, and its water surface
then lies below the cell-mean bed.

In [ ]:
function sgs_vs_li(sgs, li, t; title)
    ks, kl = FA.frame_at(sgs, t), FA.frame_at(li, t)
    hs, hl, ws, wl = FA.depth(sgs, ks), FA.depth(li, kl), FA.wse(sgs, ks), FA.wse(li, kl)
    V, zbar = FA.volume(sgs, ks), sgs.mesh.elevation
    both = [i for i in eachindex(hs) if hs[i] > 0.1 && hl[i] > 0.1 && isfinite(ws[i]) && isfinite(wl[i])]
    below = [i for i in eachindex(ws) if isfinite(ws[i]) && isfinite(zbar[i]) && ws[i] <= zbar[i] && isfinite(V[i])]
    @printf("%s: stored %.3f Mm³ (local-inertial %.3f)\n", FP.hhmm(sgs.t[ks]), FA.storage(sgs, ks) / 1e6, FA.storage(li, kl) / 1e6)
    @printf("water held below the cell-mean bed: %.3f Mm³ in %d cells\n", sum(V[below]; init = 0.0) / 1e6, length(below))
    @printf("water surface, SGS − local-inertial, over %d cells deeper than 0.1 m in both: median %+.2f m, mean %+.2f m\n",
            length(both), median(ws[both] .- wl[both]), mean(ws[both] .- wl[both]))
    @printf("deepest SGS water above a cell's lowest point: %.1f m (local-inertial: %.1f m above the cell-mean bed)\n",
            maximum(filter(isfinite, hs)), maximum(filter(isfinite, hl)))
    d = fill(NaN, length(ws)); d[both] .= ws[both] .- wl[both]
    g = sgs.mesh
    dm = FA.MeshGeom(g.ids, g.lon, g.lat, d, g.area, g.polygons, g.neighbours, g.static, g.id_scheme)
    fig = Figure()
    # FP maps draw a Run: wrap the difference as the "elevation" of a one-frame mesh run
    FP.fieldmap!(fig[1, 1], mesh_run(dm, title), 1; var = :elevation, colormap = FP.DIVERGING, colorrange = (-1, 1),
                 basemap = :grey, sources = false, width = 560, colorbar_label = "Δ water surface (m)",
                 subtitle = "t = " * FP.hhmm(sgs.t[ks]) * " · cells deeper than 0.1 m in both")
    resize_to_layout!(fig)
    return fig
end
for (n, r) in P   # removed volumes at the end of the run (55.5 h)
    n in ("a5_sgs", "a5_li", "a5_swe") && @printf("%-34s removed by 55.5 h: %.3f Mm³\n", r.label, r.info["summary"]["vol_removed_m3"] / 1e6)
end
sgs_vs_li(P["a5_sgs"], P["a5_li"], T_PEAK; title = "A5: SGS − local-inertial water surface, river only")

**The SGS excess is not sub-grid storage.** Only 0.07 Mm³ lies below the
cell-mean bed. The SGS water surface is about 3 m above the local-inertial one
(median over the 9 300 cells deeper than 0.1 m in both runs). It is more than 1 m
higher over almost the whole Eden floodplain, and close to the local-inertial
level only near the western outflow edge and in the upper Caldew and Petteril. Its water stands up to 18 m above
the cells' lowest points, and by 55.5 h it has let out 131 Mm³, against 150 Mm³
for the local-inertial run.

**This looks like a conveyance problem in the SGS solver at high flows.** In the
6 h test with rain, SGS stored 1.28 times the local-inertial run (§6). Here it
holds the Eden back by metres. With no sub-grid reference this cannot be called
wrong outright. But a water level metres above the full shallow-water solution on
the same terrain points to a problem in the SGS flow terms at high stage. **It
should be investigated before SGS results at high flows are used.**

### 5.6 What the peak window says

- **The early-window result does not carry over.** In the first 6 h with the test
  rain, local-inertial stored 1.25× SWE on A5 and 0.99× on Cartesian. Near the
  peak, with the river alone, it stores 0.76× and 0.85×. Both the sign and the size
  of its error depend on the flow.
- **Near the peak the river is deep and fast, and the local-inertial runs convey
  more.** Their deep water moves faster, they let more water out, and their water
  stands lower everywhere.
- **A plausible mechanism, not separated here from the solvers' different bed
  representations.** The local-inertial equations leave out the advection of
  momentum. The full shallow-water equations keep it, and with it the momentum
  losses where the flow bends, expands and spills onto the floodplain. Around the
  Eden's tight meanders at 1 000–1 300 m³/s, those losses could be what holds the
  reference's water higher.
- **In the early window the flow is different.** It is a small river plus thin
  rain-on-grid films, where the step bed's water held in shallow hollows dominates
  (§6). So the error had the opposite sign there.
- **The mesh.** Near the peak the local-inertial result hardly depends on the mesh,
  but the reference does: the A5 reference stores 12% more than the Cartesian
  one, its deep water is slower, and its Eden stands higher throughout.
  - The straight-channel series (HYDRAULICS §15.12) found that the SWE solver on
    A5 conveys worst in channels running 0–15° from east at that series' test
    site.
  - The Eden here runs broadly east–west. Whether that is the cause is untested.
- **What to take from it.**
  - At the peak, the local-inertial default is within 15% of the reference's
    storage on Cartesian (CSI 0.92) and within 25% on A5 (CSI 0.87).
  - A single comparison window does not characterise the solver, least of all
    one dominated by a test rain.
  - The SGS solver's behaviour at high flow (§5.5) needs investigating.

## 6. The recorded test window: the first 6 h, with test rain (`carlisle`)

This is the comparison recorded in HYDRAULICS §15.10–15.11, which set the 0.3.0
defaults. It covers the first 6 h of the event, with the river starting dry, and
**adds 10 mm/h of uniform rain on every cell**.

**That rain is an artificial test forcing, not the 2005 event's rain.** It comes
from the damping series of HYDRAULICS §15.6–15.11, where it served to test thin
sheet flow over a real floodplain. Over 6 h it puts 60 mm, 0.88 Mm³, on the whole
domain. That is only 15% of the water added (the inflows bring 5.04 Mm³), but it
wets every cell.

### 6.1 How much of the 6 h flooding is the rain?

The `*_norain` runs repeat the local-inertial runs without the rain.

In [ ]:
rows = [[g, f2(m6["$(k)_li"]["wet_area_0p1_km2"]), f2(m6["$(k)_li_norain"]["wet_area_0p1_km2"]),
         f3(m6["$(k)_li"]["storage_m3"] / 1e6), f3(m6["$(k)_li_norain"]["storage_m3"] / 1e6)]
        for (g, k) in (("A5", "a5"), ("Cartesian", "cart"))]
htmltable(["local-inertial, 6 h", "area > 0.1 m with rain (km²)", "without rain (km²)",
           "storage with rain (Mm³)", "without rain (Mm³)"], rows; caption = "carlisle at 6 h: with and without the test rain")

In [ ]:
let fig = Figure(), runs = [S["a5_li"] S["a5_li_norain"]; S["cart_li"] S["cart_li_norain"]]
    titles = ["A5 local-inertial + 10 mm/h test rain" "A5 local-inertial, river only";
              "Cartesian local-inertial + 10 mm/h test rain" "Cartesian local-inertial, river only"]
    for i in 1:2, j in 1:2
        FP.fieldmap!(fig[i, j], runs[i, j], nothing; t = T6, dry_threshold = 0.1, colorrange = (0, 5), width = 430,
                     title = titles[i, j], subtitle = "t = 06:00 · depth > 0.1 m", colorbar = j == 2)
    end
    resize_to_layout!(fig)
    fig
end

At 6 h the river inflow alone floods little more than the channels and the
lowest ground beside them. With the test rain, the flooded area (deeper than
0.1 m) is about 2.4 times larger on both meshes: hollows all over the floodplain fill with
rain. The extensive 6 h "flooding" of this case is therefore mostly rain-on-grid,
not river flooding, and its maps should not be read as the event's flood extent.

### 6.2 Local-inertial against SWE at 6 h, with the test rain

The recorded result, from the metrics of `carlisle`:

In [ ]:
rows = [[S[n].label, f3(m6[n]["storage_m3"] / 1e6), haskey(m6[n], "storage_ratio") ? f3(m6[n]["storage_ratio"]) : "–",
         haskey(m6[n], "rms_dh") ? f3(m6[n]["rms_dh"]) : "–", haskey(m6[n], "csi") ? f2(m6[n]["csi"]) : "–",
         haskey(m6[n], "bias") ? f2(m6[n]["bias"]) : "–", @sprintf("%.0f", m6[n]["arrival_s"])] for n in ORDER]
display(htmltable(["run", "storage (Mm³)", "÷ SWE", "rms Δh (m)", "CSI", "bias", "arrival (s)"], rows;
                  caption = "carlisle at 6 h (inflows + 10 mm/h test rain). Recorded (HYDRAULICS §15.10–15.11): SWE 2.690 (A5) and 2.331 (Cartesian) Mm³; local-inertial 1.25× and 0.99× SWE; SGS 4.30 Mm³. Arrival: 50 000 m³ out."))
let fig = Figure(size = (820, 460))
    FP.storage_plot!(fig[1, 1], [S[n] for n in ("a5_li", "a5_swe", "a5_sgs", "cart_li", "cart_swe")];
                     title = "Water stored, carlisle (first 6 h, inflows + 10 mm/h test rain)")
    display(fig)
end;

In [ ]:
let fig = Figure()
    FP.difference_map!(fig[1, 1], S["a5_swe"], S["a5_li"]; t = T6, colorrange = (-1.5, 1.5), width = 460,
                       title = "A5: local-inertial − SWE, 6 h, + test rain")
    FP.difference_map!(fig[1, 2], S["cart_swe"], S["cart_li"]; t = T6, colorrange = (-1.5, 1.5), width = 460,
                       title = "Cartesian: local-inertial − SWE, 6 h, + test rain")
    resize_to_layout!(fig)
    display(fig)
end;
display(storage_by_class(S["a5_swe"], S["a5_li"], T6))
display(storage_by_class(S["cart_swe"], S["cart_li"], T6))

**What the 6 h result shows:**

- **Cartesian.** The local-inertial default stays within 1% of the reference.
- **A5.** It stores 25% more, because it lets about 0.68 Mm³ less water out of the
  domain. Its first 50 000 m³ of outflow arrives about 50 min later.
- **Where the A5 excess is.** It is spread over every depth class. About a third
  is a film of about 2 cm on the 24 000 cells that the reference has nearly dry,
  which is the test rain's sheet. The rest is on the shallow floodplain and in the
  channels, which are 0.4 m deeper on average.
- **The cause, per HYDRAULICS §15.10, is mostly the bed representation.** The
  local-inertial solver lets water cross a face only above the higher of the two
  cell beds, max(z_i, z_j), so the cells form a staircase. The second-order SWE
  reference rebuilds a sloping bed at each face. A thin film drains across a
  sloping face bed, but a step bed holds it in shallow hollows. A first-order SWE
  run, which also uses a step bed, shows the same pattern of excess, more
  strongly.

With the test rain, much of this window is exactly that kind of thin sheet flow,
so the 6 h ratio of 1.25 says as much about rain-on-grid films as about the
river. §5 tests the river itself, at its peak and without rain.

## 7. Summary and pointers

**Summary.** Local-inertial against the full shallow-water reference, on the same
mesh:

| | Cartesian 20 m | A5 res 18 |
|---|---|---|
| **Near the peak (55.5 h, river only)**: storage ÷ SWE | 0.85 | 0.76 |
| rms Δh | 0.34 m | 0.54 m |
| CSI, bias | 0.92, 0.93 | 0.87, 0.89 |
| **First 6 h, with the 10 mm/h test rain**: storage ÷ SWE | 0.99 | 1.25 |
| rms Δh | 0.13 m | 0.29 m |
| CSI, bias | 0.69, 0.85 | 0.62, 1.06 |

- **The test rain dominates the first 6 h.** It is an artificial test forcing,
  not the event's rain, and it makes most of that window's flooded area: about
  2.4 times the area the river floods on its own.
- **Near the peak the local-inertial default stores less than the reference.**
  Its deep water is faster and it floods slightly less: 15% less storage on
  Cartesian and 24% less on A5. Early in the event, with the rain, its A5 run
  stored 25% *more*. The sign of its error depends on the flow.
- **The mesh.** Near the peak the local-inertial runs hardly differ between the
  meshes. The two references differ by 12%, the A5 one holding the Eden higher.
- **SGS.** It holds the Eden metres above both other solvers near the peak (3.3
  times the local-inertial storage). This needs investigating.
- **Mass.** It is conserved to round-off in every run.

**Replaying a run** in the interactive GLMakie viewer, with a frame slider and
the live viewer's controls. It needs the full stack (`--project=.`, without
`FLOODA5_NO_VIS`):

```bash
julia --project=. --threads auto visualisation/replay.jl test/case_runs/carlisle_peak/runs/a5_li.h5 --from 178200
julia --project=. --threads auto visualisation/replay.jl test/case_runs/carlisle_peak/runs/a5_swe.h5 --var velocity --from 199800
```

**From the shell:**

```bash
FLOODA5_NO_VIS=1 julia --project=. cases/run.jl show carlisle_peak            # description and commands
FLOODA5_NO_VIS=1 julia --project=. cases/run.jl run carlisle carlisle_peak     # prepare, run, check
FLOODA5_NO_VIS=1 julia --project=. cases/run.jl check carlisle_peak            # check existing runs
julia --project=. FloodModel.jl --rerun test/case_runs/carlisle_peak/runs/a5_li.h5   # reproduce one run
```

**Further reading:**

- `docs/TEST_CASES.md` §3: the Carlisle cases.
- `docs/HYDRAULICS.md`:
  - §15.10: the SWE reference and the bed-representation analysis;
  - §15.11: the 0.3.0 defaults;
  - §15.12: the channel series.
- `docs/DATA_FORMATS.md`: the output file (§2) and the `.bci`/`.bdy` formats (§3).
- `docs/USER_GUIDE.md` §6.5: the figure functions used here.